# Python and CLI for Data Science - Session 13

- *Course*: Python and CLI for Data Science
- *Session*: 13
- *Unit*: Scikit-Learn

## Scikit-Learn Introduction

### (Re)sources:
- [Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/index.html) _by Jake VanderPlas (Code released under the MIT License)_


- [Scikit-Learn](http://scikit-learn.org) provides:

    - Efficient versions many common machine learning algorithms
    - A clean, uniform, and streamlined API
    - A very useful and complete online documentation

**After learning learning the syntax of one model, switching to other models is easy.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

## Data Representation in Scikit-Learn

- In Scikit-learn data is represented as tables (matrices/vectors)
- A row represents an individual element of a dataset
- A column represents a quantity of an element

- In this example we consider the [Iris dataset](https://en.wikipedia.org/wiki/Iris_flower_data_set)

In [ ]:
from sklearn.datasets import load_iris

iris_data = load_iris()
iris = pd.DataFrame(iris_data["data"], columns=iris_data["feature_names"])
iris["species"] = iris_data["target"]
iris["species"] = iris["species"].map(lambda x: iris_data["target_names"][x])
iris

### The Features Matrix

- The input data to a model is referred to as a *features matrix*
- This matrix is often stored in a variable named `X` and is two-dimensional with shape `[n_samples, n_features]`
- Features are often real-valued, but may be Boolean or discrete-valued

In [ ]:
X_iris = iris.drop("species", axis=1)
X_iris

### The Target Array

- The desired output is called *label* or *target* array and often represented by the variable `y`
- The target array is usually one-dimensional with length `n_samples`
- The target array may have continuous numerical values, or discrete classes/labels

- The only difference between a target array and a feature column is that the target array is the quantity we want to *predict from the features* (the dependent variable)

**Target and feature columns can be interchangable**

In [ ]:
y_iris = iris.loc[:, "species"]
y_iris

### Visualizing Features and Targets

- A great option to visualize the features and targets is seaborn's `sns.pairplot` function

- It shows:
    - The distribution of features (dependent on the target variable)
    - How the features relate to one another

In [ ]:
sns.pairplot(iris, hue="species", height=1.5)

## The Estimator API

- The Scikit-Learn API's principles are outlined in the [Scikit-Learn API paper](http://arxiv.org/abs/1309.0238):

    - *Consistency*: All objects share a common interface

    - *Inspection*: All specified parameter values are exposed as public attributes

    - *Limited object hierarchy*: 
        - Only algorithms are represented by Python classes
        - Datasets are represented in standard formats (NumPy arrays, Pandas DataFrames, SciPy sparse matrices)
        - Parameter names use standard Python strings

    - *Composition*: Many machine learning tasks are expressed as sequences of more fundamental algorithms

    - *Sensible defaults*: When models require parameters, the library defines appropriate default values

**Every machine learning algorithm in Scikit-Learn is implemented consistently via the Estimator API**

### Basics of the API

- Most commonly, the steps to use the Scikit-Learn Estimator API are as follows:

    1. Choose a class of model by importing the appropriate estimator class
    2. Choose model hyperparameters by instantiating this class with desired values
    3. Fit the model to your data by calling the `fit` method of the model instance
    4. Apply the model to new data:
       - For supervised learning, often we predict labels for unknown data using the `predict` method
       - For unsupervised learning, we often transform or infer properties of the data using the `transform` or `predict` method

### Supervised Learning Example: Simple Linear Regression

- As an example, let's consider a simple linear regression—that is, the common case of fitting a line to $(x, y)$ data

In [ ]:
x = 10 * np.random.rand(50)
y = 2 * x - 5 + np.random.randn(50)
plt.scatter(x, y)

#### 1. Choose a class of model

- In Scikit-Learn, every class of model is represented by a Python class.
- If we would like to compute a simple `LinearRegression` model, we can import the linear regression class

In [ ]:
from sklearn.linear_model import LinearRegression

#### 2. Choose model hyperparameters

*A class of model is not the same as an instance of a model*.

- Depending on the model class, we might need to answer questions like the following:


    - Would we like to fit for the offset (i.e., y-intercept)?
    - Would we like the model to be normalized?
    - Would we like to preprocess our features?
    - What degree of regularization would we like to use in our model?
    - How many model components would we like to use?


- These choices are represented as *hyperparameters*
- Hyperparameters are chosen by passing values at model instantiation

- The `LinearRegression` class does not have many hyperparameters, but we can modify whether the model should learn the y-intercept

In [ ]:
model = LinearRegression(fit_intercept=True)
model

#### 3. Fit the model to the data

- We can apply the data to the model using `model.fit()`
- It causes model-dependent internal computations to take place
- The results of these computations are stored in model-specific attributes

In [ ]:
X = x.reshape(50, 1)  # convert the feature vector to a feature matrix
model.fit(X, y)

_All model parameters that were learned during the `fit` process have trailing underscores_

- For a linear regression model this is:
    - `model.coef_` (the slope)
    - `model.intercept_` (the y-intercept)

In [ ]:
model.coef_

In [ ]:
model.intercept_

#### 4. Predict labels for unknown data

- Once trained, we can use the model to predict values for new unseen data using `model.predict()`

- For example, our "new data" will be a grid of *x* values
- We again need to reshape these values into a `[n_samples, n_features]`-shaped feature matrix

In [ ]:
Xfit = np.linspace(-1, 11).reshape(50, 1)
yfit = model.predict(Xfit)

- We can visualize our predictions using `matplotlib`

In [ ]:
plt.scatter(x, y)
plt.scatter(Xfit, yfit)

## Exercise

- What happens to our model when we set `fit_intercept=False`?

In [ ]:
# Solve the exercise here

In [ ]:
model = LinearRegression(fit_intercept=False)
model.fit(X, y)
print(model.coef_)
print(model.intercept_)
yfit = model.predict(Xfit)
plt.scatter(x, y)
plt.plot(Xfit, yfit)

### Supervised Learning Example: Iris Classification

- Let's consider the Iris dataset for a more realistic example
- We train a model on a portion of the dataset and predict the species on the remaining portion
- Splitting data can be done using the `train_test_split` method

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_iris, y_iris, test_size=0.25, random_state=1)

- We will use a simple generative model known as *Gaussian naive Bayes*
- It proceeds by assuming each class is drawn from an axis-aligned Gaussian distribution
- It is often a good model to use as a baseline classification, because it is fast and has no hyperparameters

In [ ]:
from sklearn.naive_bayes import GaussianNB  # 1. choose model class
model = GaussianNB()                        # 2. instantiate model
model.fit(X_train, y_train)                 # 3. fit model to data
y_model = model.predict(X_test)             # 4. predict on new data

- We can, for example, use the `accuracy_score` method to determine how effective our classifier is

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_model)

## Exercise

- Look at the `GaussianNB` method's [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html)
- What parameters does the model learn?
- What do they tell us about the model?
- Try printing the parameters and interpreting their meaning

In [ ]:
# Solve the exercise here

In [ ]:
display(model.class_prior_, model.theta_, model.var_)

- Let us visualize the parameters that the model learns

- For every feature we plot
    - the histogram of values per species 
    - the learned distribution of the model per species

In [ ]:
from scipy.stats import norm

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
for idx, ax in enumerate(axs):
    ax.set(xlabel=model.feature_names_in_[idx])
    for c_idx, c in enumerate(model.classes_):
        bool_filter = y_iris == c
        values = X_iris.iloc[:, idx][bool_filter]
        hist = ax.hist(values, alpha=0.3, color=colors[c_idx])

        norm_x = np.linspace(values.min(), values.max(), 100)
        mu, var = model.theta_[c_idx, idx], model.var_[c_idx, idx]
        ax.plot(norm_x, norm.pdf(norm_x, mu, var), color=colors[c_idx], linestyle="--")

### Unsupervised Learning Example: Iris Dimensionality Reduction

- Unsupervised learning learns patterns in data without making use of labels
- The API for supervised and unsupervised algorithms is mostly the same
    - Instead of `predict`ing values for unseen data, unsupervised algorithms often `transform` data into a new space

- As an example, we will take a look at dimensionality reduction
    - Dimensionality reduction attempts to find a lower dimensional space that retains essential features of the data

- Here we will use *principal component analysis* (PCA)
- It reduces the the vector space to *n* components; in this case two

In [ ]:
from sklearn.decomposition import PCA  # 1. Choose the model class
model = PCA(n_components=2)            # 2. Instantiate the model
model.fit(X_iris)                      # 3. Fit to data
X_2D = model.transform(X_iris)         # 4. Transform the data

- We can now plot the results in 2D (using seaborn's `lmplot` for brevity)
- What does the plot tell us about the underlying data?

In [ ]:
iris["PCA1"] = X_2D[:, 0]
iris["PCA2"] = X_2D[:, 1]
sns.lmplot(x="PCA1", y="PCA2", data=iris, hue="species", fit_reg=False)

### Unsupervised Learning Example: Iris Clustering

- Another unsupervised algorithm is clustering which attempts to find distinct groups of data without reference to any labels


- Here we will use a *Gaussian mixture model* (GMM)
- A GMM attempts to model the data as a collection of Gaussian blobs

In [ ]:
from sklearn.mixture import GaussianMixture  # 1. Choose the model class
model = GaussianMixture(n_components=3)      # 2. Instantiate the model
model.fit(X_iris)                            # 3. Fit to data
y_gmm = model.predict(X_iris)                # 4. Determine labels

- Splitting the PCA plot by cluster provides insights into the data
- What does this plot reveal?

In [ ]:
iris["cluster"] = y_gmm
sns.lmplot(x="PCA1", y="PCA2", data=iris, hue="species", col="cluster", fit_reg=False)

### Unsupervised Learning Example: Handwritten Digits Dimensionality Reduction

- Let's look at a more interesting problem: the identification of handwritten digits
- We first download the data provided by Sciki-learn:

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
digits.images.shape

- The images data is a three-dimensional array: 1,797 samples each consisting of an 8 × 8 grid of pixels
- Let's visualize the first hundred of these

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(
    10, 10, figsize=(8, 8), subplot_kw={"xticks": [], "yticks": []}, gridspec_kw=dict(hspace=0.1, wspace=0.1)
)

for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap="binary", interpolation="nearest")
    ax.text(0.05, 0.05, str(digits.target[i]), transform=ax.transAxes, color="green")

- To work with the data we need to make it two-dimensional `[n_samples, n_features]` 
- We can treat each pixel in the image as a feature
    - We flatten out the pixel arrays
    - An image is reprensented by a 64-dimensional vector of pixels
    - (What problems does this representation have?)
- We also need the target array which contains the label for each digit
- Both X and y are built into the digits dataset as `data` and `target`, respectively

In [ ]:
X = digits.data
y = digits.target
X.shape, y.shape

- Visualizing a 64-dimensional parameter space is difficult
- Instead, we'll reduce the number of dimensions using an unsupervised method
- We will use a manifold learning algorithm called `Isomap` and `transform` the data to two dimensions

In [ ]:
from sklearn.manifold import Isomap

iso = Isomap(n_components=2)
iso.fit(digits.data)
data_projected = iso.transform(digits.data)
print(data_projected.shape)

- The data is now two-dimensional
- Let's plot this data to see if we can learn anything from its structure
- What information can we gain from this plot?
    - Are the digits separable from one another?
    - How effective will a classifier trained on this data be?
    - Which numbers might be more difficult to detect than others?

In [ ]:
plt.scatter(data_projected[:, 0], data_projected[:, 1], c=digits.target, cmap=plt.colormaps["tab10"])
plt.colorbar(label="digit label", ticks=range(10))
plt.clim(-0.5, 9.5)

### Supervised Learning Example: Classifying Digits

- Let's check our intiution and train a `GaussianNB` model to classify digits
- We first split the data into train and test splits
- We then train the model and compute the accuracy

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

model = GaussianNB()
model.fit(X_train, y_train)
y_model = model.predict(X_test)
accuracy_score(y_test, y_model)

- With this very simple model, we achieve 83% accuracy
- This single number doesn't tell us where the model is incorrect
- One nice way to do this is to use the *confusion matrix*
- Which digits does the model tend to get wrong?

In [ ]:
from sklearn.metrics import confusion_matrix

mat = confusion_matrix(y_test, y_model)

sns.heatmap(mat, square=True, annot=True, cbar=False, cmap="Blues")
plt.xlabel("predicted value")
plt.ylabel("true value");